# Dummy Classifier

In this notebook, I will use `DummyClassifier` to classify every customer as a churner. I will calculate metrics using cross-validation to establish a baseline for comparison with other models.

Metrics:
- Accuracy
- Precision
- Recall
- F1

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import LabelEncoder

from telco_churn.config import (
    TARGET_COLUMN,
    ID_COLUMN,
)
from telco_churn.paths import TELCO_CUSTOMERS_TRAIN_PATH

In [17]:
# Define metrics for cross-validation and summary statistics
METRICS = ['accuracy', 'precision', 'recall', 'f1']

## Load the training set and separate features from the target

In [2]:
# Load the training dataset
df_train = pd.read_csv(TELCO_CUSTOMERS_TRAIN_PATH)

In [3]:
# Display first three rows to inspect the loaded data
df_train.head(3)

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,2926-JEJJC,Female,0,No,No,8,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Bank transfer (automatic),20.20,140.95,No
1,6029-CSMJE,Male,0,No,No,17,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,25.65,440.20,No
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,No


In [8]:
# Prepare training features by dropping the identifier (customer_id) and target (churn) columns
X_train = df_train.drop(
    columns=[
        TARGET_COLUMN,
        ID_COLUMN,
    ]
)

In [9]:
# Prepare the training target (churn)
y_train = df_train[TARGET_COLUMN].copy()

In [10]:
# Encode the training target as 0 (No) and 1 (Yes)
target_encoder = LabelEncoder()
y_train_prep = target_encoder.fit_transform(y_train)

In [12]:
# Confirm the target class mapping
target_encoder.classes_

array(['No', 'Yes'], dtype=object)

## DummyClassifier, cross-validation and metrics

In [13]:
# Configure DummyClassifier to classify every customer as a churner
dummy_classifier = DummyClassifier(
    strategy='constant',
    constant=1,
)

In [18]:
# Evaluate DummyClassifier using 5-fold cross-validatio
dummy_classifier_cv_results = cross_validate(
    dummy_classifier,
    X_train,
    y_train_prep,
    cv=5,
    scoring=METRICS,
)

In [22]:
dummy_mean_std_scores_dict = {}

# Calculate the mean and standard deviation for each metric
for metric in METRICS:
    dummy_mean_std_scores_dict[metric] = {
        'mean': dummy_classifier_cv_results[f'test_{metric}'].mean().round(4),
        'std': dummy_classifier_cv_results[f'test_{metric}'].std().round(4),
    }

# Create a summary table with metrics as rows and statistics as columns
dummy_metrics_stats = pd.DataFrame(dummy_mean_std_scores_dict).T
dummy_metrics_stats.index.name = 'metrics'
dummy_metrics_stats.columns.name = 'statistics'

dummy_metrics_stats

statistics,mean,std
metrics,,
accuracy,0.2654,0.0001
precision,0.2654,0.0001
recall,1.0000,0.0000
f1,0.4194,0.0001
